# Tabular Foundation Model vs 기존 머신러닝 — 조건별 비교

**연구 질문**: 데이터 크기, 결측률, 범주형 비율, 클래스 불균형에 따라 사전학습 모델(Kumo Tabular, TabICLv2)과 부스팅 모델(XGBoost·LightGBM·CatBoost)의 우위가 어떻게 달라지는가?

**목표 산출물**: "새 모델이 더 좋다"가 아니라 **어떤 데이터에서 어떤 모델을 선택해야 하는가** (결정 가이드).

| 구분 | 내용 |
|---|---|
| 과제 | 이진 분류 (양성 = 소수 클래스) |
| 사전학습 모델 | NVIDIA **Kumo Tabular** (`sdm`), **TabICLv2** (`sdm`) — 과제별 가중치 업데이트 없이 레이블된 학습 행을 문맥(context)으로 예측 |
| 기존 ML | XGBoost / LightGBM / CatBoost × {기본값, **튜닝(명시적 Optuna 예산)**}, Logistic Regression |
| 공정성 | 모든 모델이 **동일한 분할·동일한 조건 변환** 데이터를 받음. 분할 인덱스는 `results/splits/`에 저장 |
| 평가 | 성능(ROC-AUC, PR-AUC, Accuracy), 확률 보정(Log-loss, Brier, ECE), 비용(준비+튜닝+학습+추론 시간) |

**실험 구성**
- **Part A — 자연 상태 벤치마크**: 공개 데이터셋 여러 개를 원래 상태로 비교 → 데이터 메타특성과 우위의 관계
- **Part B — 통제 실험**: 기준 데이터셋에 *한 번에 한 축만* 바꿔서(크기 / 결측 / 범주형 / 불균형) 우위가 어떻게 이동하는지 측정

> Colab: 런타임 → 런타임 유형 변경 → **GPU** 선택 후 실행하세요. 결과는 실행 중에 CSV로 누적 저장되며, 세션이 끊겨도 같은 셀을 다시 실행하면 이어서 진행합니다.

In [ ]:
# ── 0. 설치 ──────────────────────────────────────────────────────────────
import importlib.util, subprocess, sys
def _ensure(pkg, spec=None):
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", spec or pkg], check=True)
_ensure("sdm", "git+https://github.com/NVIDIA/structured-data-models.git")
for p in ["xgboost", "lightgbm", "catboost", "optuna"]:
    _ensure(p)

In [ ]:
# ── 1. 설정 ──────────────────────────────────────────────────────────────
import os, time, json, math, warnings, gc
from pathlib import Path
import numpy as np, pandas as pd, torch

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")

# "quick": 세미나 데모용 (~20-30분, GPU)   "full": 결론용 (수 시간, 이어서 실행 가능)
MODE = os.environ.get("TFM_MODE", "quick")

# 결과 저장 위치. Colab에서 Drive에 저장하려면 USE_DRIVE=True
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive; drive.mount("/content/drive")
    OUT_DIR = Path("/content/drive/MyDrive/tfm_seminar/results")
else:
    OUT_DIR = Path(os.environ.get("TFM_OUT", "results"))
(OUT_DIR / "splits").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "figures").mkdir(parents=True, exist_ok=True)

# OpenML data_id. 양성 클래스 = 소수 클래스
DATASETS = {
    "blood-transfusion": 1464, "credit-approval": 29, "diabetes": 37, "credit-g": 31,
    "qsar-biodeg": 1494, "kc1": 1067, "ozone-level": 1487, "spambase": 44,
    "churn": 40701, "phoneme": 1489, "bank-marketing": 1461, "adult": 1590,
}

PRESETS = {
    "quick": dict(
        part_a_datasets=["blood-transfusion", "credit-g", "kc1", "phoneme", "churn", "adult"],
        part_b_datasets=["adult", "phoneme"],
        seeds=[0], tune_trials=10, kumo_size="large" if DEVICE.type == "cuda" else "small",
        num_estimators=8,
    ),
    "full": dict(
        part_a_datasets=list(DATASETS),
        part_b_datasets=["adult", "bank-marketing", "phoneme", "churn"],
        seeds=[0, 1, 2], tune_trials=30, kumo_size="large", num_estimators=8,
    ),
}
CFG = PRESETS[MODE]

# Part A: 자연 상태. 학습 행 상한 / 테스트 행 상한
A_MAX_TRAIN, A_MAX_TEST = 5000, 2000
# Part B: 기준 조건 + 한 축씩 변경
B_BASE = dict(n_train=1000, missing=0.0, cat_frac=0.0, prevalence=None)  # prevalence None = 원래 비율
B_AXES = {
    "size":       [100, 300, 1000, 3000, 10000],
    "missing":    [0.0, 0.1, 0.3, 0.5],          # 학습·테스트 X에 MCAR 결측 주입 비율
    "cat_frac":   [0.0, 0.5, 1.0],               # 수치형 열 중 범주형(10분위 구간)으로 바꾸는 비율
    "prevalence": [None, 0.10, 0.03],            # 양성 비율 (학습·테스트 동일)
}
B_N_TEST = 2000
CAT_BINS = 10

MODELS = ["Kumo", "TabICLv2", "XGB-default", "XGB-tuned", "LGBM-default", "LGBM-tuned",
          "CatBoost-default", "CatBoost-tuned", "LogReg"]
FM_MODELS = ["Kumo", "TabICLv2"]
RESULTS_CSV = OUT_DIR / f"results_{MODE}.csv"
print(MODE, CFG)

## 2. 데이터 로딩과 조건 변환

- 범주형은 `category`, 수치형은 `float`로 통일하고, 결측은 그대로 둡니다 (각 모델이 자체적으로 처리).
- **조건 변환 순서**: (1) 양성 비율 재표집 → (2) 층화 test/train 분할 → (3) 수치형→범주형 구간화(**train 분위수로** 경계 결정) → (4) MCAR 결측 주입(train·test 독립).
- 같은 `(dataset, 조건, seed)`이면 항상 같은 인덱스·같은 마스크가 나오고, 인덱스는 `splits/`에 저장됩니다.
- ⚠️ `cat_frac` 축은 수치형을 구간화하므로 "범주형 비율 증가"와 "정보 손실"이 함께 일어납니다. 해석할 때 이 점을 감안해야 합니다.

In [ ]:
import hashlib
from sklearn.datasets import fetch_openml

def _stable_seed(key):
    return int(hashlib.md5(key.encode()).hexdigest()[:8], 16)

_DATA_CACHE = {}
def load_dataset(name):
    if name in _DATA_CACHE:
        return _DATA_CACHE[name]
    d = fetch_openml(data_id=DATASETS[name], as_frame=True, parser="auto")
    X, y_raw = d.data.copy(), d.target.astype(str)
    pos = y_raw.value_counts().idxmin()
    y = (y_raw == pos).astype(int).to_numpy()
    for c in X.columns:
        if X[c].dtype.kind in "biuf" and X[c].dtype.name != "category":
            X[c] = X[c].astype(float)
        else:
            s = X[c].astype(object)
            X[c] = s.where(s.isna(), s.astype(str)).astype("category")
    X.columns = [str(c) for c in X.columns]
    _DATA_CACHE[name] = (X.reset_index(drop=True), y)
    return _DATA_CACHE[name]

def meta_features(X, y):
    cat = [c for c in X if X[c].dtype.name == "category"]
    return dict(n_rows=len(X), n_feat=X.shape[1], cat_ratio=len(cat) / X.shape[1],
                missing_rate=float(X.isna().mean().mean()), prevalence=float(y.mean()))

def _stratified_take(idx, y, n, rng):
    # idx 중 n개를 양성 비율을 유지하며 뽑는다
    pos, neg = idx[y[idx] == 1], idx[y[idx] == 0]
    n_pos = int(round(n * len(pos) / len(idx)))
    n_pos = min(max(n_pos, 2), len(pos)); n_neg = min(n - n_pos, len(neg))
    return np.concatenate([rng.choice(pos, n_pos, replace=False), rng.choice(neg, n_neg, replace=False)])

def make_task(name, cond, seed):
    # cond: dict(n_train, n_test, missing, cat_frac, prevalence). 불가능하면 None
    X, y = load_dataset(name)
    key = f"{name}|{json.dumps(cond, sort_keys=True)}|{seed}"
    rng = np.random.default_rng(_stable_seed(key))
    idx = np.arange(len(y))
    if cond["prevalence"] is not None:          # (1) 양성 비율 재표집: 음성 전체 유지, 양성만 줄임
        p = cond["prevalence"]; pos, neg = idx[y == 1], idx[y == 0]
        n_pos = int(len(neg) * p / (1 - p))
        if n_pos > len(pos):
            return None
        idx = np.concatenate([rng.choice(pos, n_pos, replace=False), neg])
    if cond["n_train"] + cond["n_test"] > len(idx):
        return None
    te = _stratified_take(idx, y, cond["n_test"], rng)            # (2) 분할
    tr = _stratified_take(np.setdiff1d(idx, te), y, cond["n_train"], rng)
    rng.shuffle(tr)
    np.savez_compressed(OUT_DIR / "splits" / f"{_stable_seed(key)}.npz", train=tr, test=te)
    Xtr, Xte = X.iloc[tr].reset_index(drop=True).copy(), X.iloc[te].reset_index(drop=True).copy()
    num_cols = [c for c in X if X[c].dtype.name != "category"]
    k = int(round(cond["cat_frac"] * len(num_cols)))              # (3) 수치형 → 범주형
    for c in rng.choice(num_cols, k, replace=False) if k else []:
        edges = np.unique(np.nanquantile(Xtr[c], np.linspace(0, 1, CAT_BINS + 1)[1:-1]))
        for D in (Xtr, Xte):
            codes = np.searchsorted(edges, D[c].to_numpy(), side="right")
            lab = pd.Series([f"bin{v}" for v in codes], dtype=object).where(D[c].notna().to_numpy())
            D[c] = pd.Categorical(lab, categories=[f"bin{i}" for i in range(len(edges) + 1)])
    if cond["missing"] > 0:                                       # (4) MCAR 결측
        for D in (Xtr, Xte):
            mask = rng.random(D.shape) < cond["missing"]
            for j, c in enumerate(D.columns):
                D.loc[mask[:, j], c] = np.nan
    return Xtr, y[tr], Xte, y[te]

def level_label(v):
    return "orig" if v is None else str(v)

def task_list():
    tasks = []
    for name in CFG["part_a_datasets"]:
        X, y = load_dataset(name)
        n_test = min(A_MAX_TEST, int(len(y) * 0.3)); n_train = min(A_MAX_TRAIN, len(y) - n_test)
        for s in CFG["seeds"]:
            tasks.append(dict(part="A", dataset=name, axis="natural", level="natural", seed=s,
                              cond=dict(n_train=n_train, n_test=n_test, missing=0.0, cat_frac=0.0, prevalence=None)))
    for name in CFG["part_b_datasets"]:
        for axis, levels in B_AXES.items():
            for lv in levels:
                cond = dict(B_BASE, n_test=B_N_TEST); cond["n_train" if axis == "size" else axis] = lv
                for s in CFG["seeds"]:
                    tasks.append(dict(part="B", dataset=name, axis=axis, level=level_label(lv), seed=s, cond=cond))
    return tasks

TASKS = task_list()
print(len(TASKS), "tasks")
pd.DataFrame([dict(dataset=n, **meta_features(*load_dataset(n))) for n in
              dict.fromkeys(CFG["part_a_datasets"] + CFG["part_b_datasets"])]).round(3)

## 3. 평가 지표

- **성능**: ROC-AUC, PR-AUC(불균형에 민감), Accuracy(임계값 0.5)
- **확률 보정**: Log-loss, Brier score, ECE(15구간, 동일 폭)
- **비용**: `prep`(형 변환·텐서화) + `tune`(하이퍼파라미터 탐색) + `fit`(학습 / FM은 문맥 인코딩·KV 캐시) + `predict`. FM 가중치 다운로드·로딩 시간은 **1회성**이라 `load_s`에 따로 기록하고 합계에서는 뺍니다.

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss, brier_score_loss, accuracy_score

def ece_score(y, p, bins=15):
    edges = np.linspace(0, 1, bins + 1); idx = np.clip(np.digitize(p, edges) - 1, 0, bins - 1)
    return float(sum(abs(y[idx == b].mean() - p[idx == b].mean()) * (idx == b).mean()
                     for b in range(bins) if (idx == b).any()))

def evaluate(y, p):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    return dict(auc=roc_auc_score(y, p), pr_auc=average_precision_score(y, p),
                acc=accuracy_score(y, p >= 0.5), logloss=log_loss(y, p, labels=[0, 1]),
                brier=brier_score_loss(y, p), ece=ece_score(y, p))

## 4. 모델

### 4-1. Tabular Foundation Model (Kumo Tabular, TabICLv2)
`sdm.TableTensor`로 변환한 뒤 `fit`(문맥 캐시)과 `predict`을 호출합니다. 가중치 업데이트는 없습니다. 레이블은 학습 행에서 문맥으로 들어갑니다.

In [ ]:
import sdm

def _sync():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

_FM_CACHE, FM_LOAD_S = {}, {}
def get_fm(name):
    if name not in _FM_CACHE:
        t = time.perf_counter()
        if name == "Kumo":
            _FM_CACHE[name] = sdm.models.KumoTabular(task="classification", size=CFG["kumo_size"], device=DEVICE)
        else:
            _FM_CACHE[name] = sdm.models.TabICLv2(task="classification", device=DEVICE)
        _sync(); FM_LOAD_S[name] = time.perf_counter() - t
    return _FM_CACHE[name]

def run_fm(name, Xtr, ytr, Xte, seed):
    model = get_fm(name)
    t0 = time.perf_counter()
    stypes = sdm.infer_stypes(Xtr)
    x_tr = sdm.TableTensor.from_pandas(df=Xtr, stypes=stypes, device=DEVICE)
    x_te = sdm.TableTensor.from_pandas(df=Xte, stypes=stypes, device=DEVICE)
    y_tr = sdm.TableTensor.from_pandas(df=pd.DataFrame({"y": ytr.astype(str)}), stypes={"y": "categorical"}, device=DEVICE)
    _sync(); t1 = time.perf_counter()
    gen = torch.Generator(DEVICE).manual_seed(seed)
    with torch.inference_mode(), torch.amp.autocast(DEVICE.type, torch.float16, enabled=DEVICE.type == "cuda"):
        model.fit(x=x_tr, y=y_tr, num_estimators=CFG["num_estimators"], estimator_batch_size=1, generator=gen)
        _sync(); t2 = time.perf_counter()
        # 테스트 행을 나눠서 예측해 메모리를 아낀다
        parts = []
        for s in range(0, len(Xte), 1000):
            out = model.predict(x_te[s:s + 1000])
            parts.append(out.numerical[..., out.columns[sdm.Stype.numerical].index("1")].float().cpu().numpy())
        _sync(); t3 = time.perf_counter()
    model.clear()
    return np.concatenate(parts), dict(prep_s=t1 - t0, tune_s=0.0, fit_s=t2 - t1, predict_s=t3 - t2)

### 4-2. 부스팅 모델과 로지스틱 회귀

- **default**: 라이브러리 기본 하이퍼파라미터, 범주형은 각 라이브러리의 네이티브 처리 기능 사용
- **tuned**: Optuna TPE, `tune_trials`회의 **명시적 예산**. 목적함수는 검증 log-loss이고 early stopping은 50라운드입니다. 학습 행이 1000개 이하면 3-fold CV, 그보다 많으면 80/20 홀드아웃으로 검증합니다. 찾은 하이퍼파라미터와 반복 수로 학습 데이터 전체에 다시 학습합니다.
- **LogReg**: 결측 대체 + 원-핫 + 표준화 파이프라인

In [ ]:
import xgboost as xgb, lightgbm as lgb, catboost as cb, optuna
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
optuna.logging.set_verbosity(optuna.logging.WARNING)
N_JOBS = int(os.environ.get("TFM_N_JOBS", os.cpu_count()))  # macOS 로컬에서는 OpenMP 충돌 때문에 1 권장

def _cat_cols(X):
    return [c for c in X if X[c].dtype.name == "category"]

def _for_catboost(X):
    X = X.copy()
    for c in _cat_cols(X):
        X[c] = X[c].astype(object).where(X[c].notna(), "__NA__").astype(str)
    return X

def _build(lib, params, n_rounds, seed):
    if lib == "XGB":
        return xgb.XGBClassifier(**params, n_estimators=n_rounds, tree_method="hist", enable_categorical=True,
                                 max_cat_to_onehot=1, random_state=seed, n_jobs=N_JOBS, verbosity=0)
    if lib == "LGBM":
        return lgb.LGBMClassifier(**params, n_estimators=n_rounds, random_state=seed, n_jobs=N_JOBS, verbose=-1)
    return cb.CatBoostClassifier(**params, iterations=n_rounds, random_seed=seed, thread_count=N_JOBS, verbose=0)

def _fit(lib, model, X, y, Xval=None, yval=None):
    cat = _cat_cols(X)
    if lib == "CatBoost":
        X = _for_catboost(X); Xval = None if Xval is None else _for_catboost(Xval)
        kw = dict(cat_features=cat)
        if Xval is not None: kw.update(eval_set=(Xval, yval), early_stopping_rounds=50)
        return model.fit(X, y, **kw)
    if Xval is None:
        return model.fit(X, y)
    if lib == "XGB":
        model.set_params(early_stopping_rounds=50)
        return model.fit(X, y, eval_set=[(Xval, yval)], verbose=False)
    return model.fit(X, y, eval_set=[(Xval, yval)], callbacks=[lgb.early_stopping(50, verbose=False)])

def _best_iter(lib, model):
    if lib == "XGB": return model.best_iteration + 1
    if lib == "LGBM": return model.best_iteration_ or model.n_estimators
    return model.get_best_iteration() + 1

def _space(lib, trial):
    if lib == "XGB":
        return dict(learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
                    max_depth=trial.suggest_int("max_depth", 2, 10),
                    min_child_weight=trial.suggest_float("min_child_weight", 1e-2, 20, log=True),
                    subsample=trial.suggest_float("subsample", 0.5, 1.0),
                    colsample_bytree=trial.suggest_float("colsample_bytree", 0.4, 1.0),
                    reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
                    reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 10, log=True))
    if lib == "LGBM":
        return dict(learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
                    num_leaves=trial.suggest_int("num_leaves", 4, 128, log=True),
                    min_child_samples=trial.suggest_int("min_child_samples", 2, 100, log=True),
                    subsample=trial.suggest_float("subsample", 0.5, 1.0), subsample_freq=1,
                    colsample_bytree=trial.suggest_float("colsample_bytree", 0.4, 1.0),
                    reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
                    min_data_per_group=trial.suggest_int("min_data_per_group", 2, 100, log=True),
                    cat_smooth=trial.suggest_float("cat_smooth", 1, 50, log=True))
    return dict(learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
                depth=trial.suggest_int("depth", 3, 10),
                l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 1, 30, log=True),
                random_strength=trial.suggest_float("random_strength", 1e-2, 10, log=True),
                bagging_temperature=trial.suggest_float("bagging_temperature", 0, 1))

def _tune(lib, Xtr, ytr, seed):
    if len(ytr) <= 1000:
        folds = list(StratifiedKFold(3, shuffle=True, random_state=seed).split(Xtr, ytr))
    else:
        folds = [train_test_split(np.arange(len(ytr)), test_size=0.2, stratify=ytr, random_state=seed)]
    def objective(trial):
        params = _space(lib, trial); losses, iters = [], []
        for a, b in folds:
            m = _fit(lib, _build(lib, params, 2000, seed), Xtr.iloc[a], ytr[a], Xtr.iloc[b], ytr[b])
            p = m.predict_proba(_for_catboost(Xtr.iloc[b]) if lib == "CatBoost" else Xtr.iloc[b])[:, 1]
            losses.append(log_loss(ytr[b], np.clip(p, 1e-6, 1 - 1e-6), labels=[0, 1])); iters.append(_best_iter(lib, m))
        trial.set_user_attr("n_rounds", int(np.mean(iters)))
        return float(np.mean(losses))
    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=seed))
    study.optimize(objective, n_trials=CFG["tune_trials"])
    return study.best_params, max(10, study.best_trial.user_attrs["n_rounds"])

_DEFAULT_ROUNDS = {"XGB": 100, "LGBM": 100, "CatBoost": 1000}   # 각 라이브러리 기본값

def run_gbdt(name, Xtr, ytr, Xte, seed):
    lib, mode = name.split("-")
    t0 = time.perf_counter()
    params, n_rounds = _tune(lib, Xtr, ytr, seed) if mode == "tuned" else ({}, _DEFAULT_ROUNDS[lib])
    t1 = time.perf_counter()
    m = _fit(lib, _build(lib, params, n_rounds, seed), Xtr, ytr)
    t2 = time.perf_counter()
    p = m.predict_proba(_for_catboost(Xte) if lib == "CatBoost" else Xte)[:, 1]
    t3 = time.perf_counter()
    return p, dict(prep_s=0.0, tune_s=t1 - t0, fit_s=t2 - t1, predict_s=t3 - t2)

def run_logreg(Xtr, ytr, Xte, seed):
    cat = _cat_cols(Xtr); num = [c for c in Xtr if c not in cat]
    to_obj = lambda X: X.astype({c: object for c in cat})
    pre = ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), num),
        ("cat", make_pipeline(SimpleImputer(strategy="constant", fill_value="__NA__"),
                              OneHotEncoder(handle_unknown="ignore", min_frequency=5)), cat)])
    model = make_pipeline(pre, LogisticRegression(max_iter=2000, C=1.0))
    t0 = time.perf_counter(); Xtr_o, Xte_o = to_obj(Xtr), to_obj(Xte); t1 = time.perf_counter()
    model.fit(Xtr_o, ytr); t2 = time.perf_counter()
    p = model.predict_proba(Xte_o)[:, 1]; t3 = time.perf_counter()
    return p, dict(prep_s=t1 - t0, tune_s=0.0, fit_s=t2 - t1, predict_s=t3 - t2)

def run_model(name, Xtr, ytr, Xte, seed):
    if name in FM_MODELS: return run_fm(name, Xtr, ytr, Xte, seed)
    if name == "LogReg": return run_logreg(Xtr, ytr, Xte, seed)
    return run_gbdt(name, Xtr, ytr, Xte, seed)

## 5. 실험 실행 (이어서 실행 가능)

각 `(task, model)` 결과를 끝날 때마다 CSV에 한 줄씩 추가합니다. 이미 완료된 조합은 건너뜁니다. 테스트 예측 확률은 보정 그래프용으로 `preds_*.parquet`에 저장합니다.

In [ ]:
def _run_key(t, m): return f'{t["part"]}|{t["dataset"]}|{t["axis"]}|{t["level"]}|{t["seed"]}|{m}'

RESULT_COLS = ["key", "model", "status", "auc", "pr_auc", "acc", "logloss", "brier", "ece",
               "prep_s", "tune_s", "fit_s", "predict_s", "total_s", "load_s", "part", "dataset", "axis", "level", "seed",
               "meta_n_rows", "meta_n_feat", "meta_cat_ratio", "meta_missing_rate", "meta_prevalence"]
prev = pd.read_csv(RESULTS_CSV) if RESULTS_CSV.exists() else pd.DataFrame(columns=RESULT_COLS)
done = set(prev.loc[prev["status"] == "ok", "key"])   # 에러난 조합은 다시 시도
pred_rows = []
start = time.perf_counter()
for i, t in enumerate(TASKS):
    todo = [m for m in MODELS if _run_key(t, m) not in done]
    if not todo:
        continue
    data = make_task(t["dataset"], t["cond"], t["seed"])
    if data is None:
        print(f"[skip] {t['dataset']} {t['axis']}={t['level']}: 행 부족")
        continue
    Xtr, ytr, Xte, yte = data
    meta = meta_features(Xtr, ytr)
    for m in todo:
        try:
            p, cost = run_model(m, Xtr, ytr, Xte, t["seed"])
            row = dict(key=_run_key(t, m), model=m, status="ok", **evaluate(yte, p), **cost)
            if t["part"] == "B" and t["seed"] == CFG["seeds"][0]:
                pred_rows.append(pd.DataFrame(dict(key=row["key"], y=yte, p=p)))
        except Exception as e:  # OOM 등은 기록하고 계속
            row = dict(key=_run_key(t, m), model=m, status=f"error: {type(e).__name__}: {str(e)[:120]}")
            if DEVICE.type == "cuda": torch.cuda.empty_cache()
        row.update({k: t[k] for k in ["part", "dataset", "axis", "level", "seed"]}, **{f"meta_{k}": v for k, v in meta.items()})
        row["total_s"] = sum(row.get(k, 0.0) for k in ["prep_s", "tune_s", "fit_s", "predict_s"])
        row["load_s"] = FM_LOAD_S.get(m, 0.0)
        pd.DataFrame([row]).reindex(columns=RESULT_COLS).to_csv(RESULTS_CSV, mode="a", header=not RESULTS_CSV.exists(), index=False)
    if pred_rows:
        pd.concat(pred_rows).to_parquet(OUT_DIR / f"preds_{MODE}_{i:04d}.parquet"); pred_rows = []
    print(f"[{i + 1}/{len(TASKS)}] {t['dataset']:16s} {t['axis']}={t['level']:<7s} seed={t['seed']}  "
          f"elapsed {(time.perf_counter() - start) / 60:.1f}m")
gc.collect()

## 6. 결과 분석

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid", context="notebook")
R = pd.read_csv(RESULTS_CSV, dtype={"level": str}, keep_default_na=False, na_values=[""])
R = R[R["status"] == "ok"].drop_duplicates("key", keep="last").copy()   # 재시도된 조합은 마지막 성공만
print(R["model"].value_counts().to_dict())
GBDT_TUNED = ["XGB-tuned", "LGBM-tuned", "CatBoost-tuned"]
GBDT_ALL = [f"{lib}-{m}" for lib in ["XGB", "LGBM", "CatBoost"] for m in ["default", "tuned"]]
PALETTE = {"Kumo": "#2a6fdb", "TabICLv2": "#7aa6f0", "XGB-default": "#f2b38a", "XGB-tuned": "#e0631d",
           "LGBM-default": "#a8d5a2", "LGBM-tuned": "#2e8b3d", "CatBoost-default": "#d8b4e2",
           "CatBoost-tuned": "#8e44ad", "LogReg": "#888888"}
ORDER = [m for m in MODELS if m in R["model"].unique()]

TASK_COLS = ["part", "dataset", "axis", "level", "seed"]

### 6-1. Part A — 자연 상태 데이터셋: 평균 순위와 지표
데이터셋별로 순위를 매긴 뒤 평균을 냅니다(낮을수록 좋음). AUC는 높을수록, log-loss·ECE·시간은 낮을수록 좋습니다.

In [ ]:
A = R[R["part"] == "A"]
def avg_rank(df, metric, higher=True):
    g = df.groupby(["dataset", "seed"])[metric].rank(ascending=not higher)
    return df.assign(rank=g).groupby("model")["rank"].mean()
summary_a = pd.DataFrame({
    "AUC": A.groupby("model")["auc"].mean(), "rank(AUC)": avg_rank(A, "auc"),
    "LogLoss": A.groupby("model")["logloss"].mean(), "rank(LogLoss)": avg_rank(A, "logloss", False),
    "ECE": A.groupby("model")["ece"].mean(), "총시간(s, 중앙값)": A.groupby("model")["total_s"].median(),
}).loc[ORDER].round(3).sort_values("rank(LogLoss)")
summary_a.to_csv(OUT_DIR / f"summary_partA_{MODE}.csv"); summary_a

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, (metric, label) in zip(axes, [("auc", "ROC-AUC (↑)"), ("logloss", "Log-loss (↓)")]):
    piv = A.groupby(["dataset", "model"])[metric].mean().unstack()[ORDER]
    gb = piv[[c for c in GBDT_ALL if c in piv]]
    best_gb = gb.max(axis=1) if metric == "auc" else gb.min(axis=1)
    order = A.groupby("dataset")["meta_n_rows"].first().sort_values().index
    for m in ["Kumo", "TabICLv2", "LogReg"] + GBDT_TUNED:
        if m in piv: ax.plot(range(len(order)), (piv[m] - best_gb).loc[order], "o-", label=m, color=PALETTE[m])
    ax.axhline(0, color="k", lw=0.8)
    ax.set_xticks(range(len(order))); ax.set_xticklabels(order, rotation=40, ha="right")
    ax.set_title(f"{label}: model − best of 6 GBDT (datasets sorted by #rows)")
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT_DIR / "figures" / f"partA_delta_{MODE}.png", dpi=160); plt.show()

### 6-2. Part B — 조건별 우위 이동
축마다 **모델 − (부스팅 6종(기본값·튜닝) 중 해당 task 최고값)**의 차이를 그립니다. 0보다 위(AUC) 또는 아래(log-loss)면 그 모델이 더 좋습니다.
비교 기준인 '부스팅 최고값'은 테스트 결과를 보고 사후에 고른 값이라 부스팅에 **유리한 보수적 기준**입니다. (학습 행이 아주 적으면 튜닝이 오히려 기본값보다 나쁠 수 있어 기본값도 포함합니다)

In [ ]:
B = R[R["part"] == "B"].copy()
def level_order(axis):
    return [level_label(v) for v in B_AXES[axis]]
best = (B[B["model"].isin(GBDT_ALL)].groupby(TASK_COLS)
        .agg(best_auc=("auc", "max"), best_logloss=("logloss", "min"), best_prauc=("pr_auc", "max")).reset_index())
BD = B.merge(best, on=TASK_COLS)
BD["d_auc"] = BD["auc"] - BD["best_auc"]; BD["d_logloss"] = BD["logloss"] - BD["best_logloss"]
BD["d_prauc"] = BD["pr_auc"] - BD["best_prauc"]

AXIS_LABEL = {"size": "train rows", "missing": "missing rate (MCAR)", "cat_frac": "numeric→categorical fraction", "prevalence": "positive rate"}
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for j, axis in enumerate(B_AXES):
    d = BD[BD["axis"] == axis]; lv = level_order(axis)
    for i, (metric, lab) in enumerate([("d_auc", "Δ AUC (↑ better)"), ("d_logloss", "Δ Log-loss (↓ better)")]):
        ax = axes[i, j]
        for m in ["Kumo", "TabICLv2", "LogReg"]:
            s = d[d["model"] == m].groupby("level")[metric].agg(["mean", "sem"]).reindex(lv)
            ax.errorbar(range(len(lv)), s["mean"], yerr=s["sem"].fillna(0), marker="o", capsize=3, label=m, color=PALETTE[m])
        ax.axhline(0, color="k", lw=0.8); ax.set_xticks(range(len(lv))); ax.set_xticklabels(lv)
        ax.set_title(f"{AXIS_LABEL[axis]}" if i == 0 else ""); ax.set_ylabel(lab if j == 0 else "")
axes[0, 0].legend(fontsize=8)
plt.suptitle("Model − best of 6 GBDT variants per condition (mean ± SE over datasets × seeds)")
plt.tight_layout(); plt.savefig(OUT_DIR / "figures" / f"partB_axes_{MODE}.png", dpi=160); plt.show()

### 6-3. 확률 보정
ECE가 낮고 reliability 곡선이 대각선에 가까울수록 "예측 확률 = 실제 빈도"에 가깝습니다. 의사결정에 확률을 그대로 쓴다면(위험 점수, 기대값 계산 등) 순위 지표(AUC)보다 이쪽이 중요합니다.

In [ ]:
cal = R.groupby("model")[["ece", "brier", "logloss"]].mean().loc[ORDER].round(4)
display(cal.sort_values("ece"))
preds = pd.concat([pd.read_parquet(f) for f in sorted(OUT_DIR.glob(f"preds_{MODE}_*.parquet"))])
preds["model"] = preds["key"].str.split("|").str[-1]
base = preds[preds["key"].str.contains(r"\|size\|1000\|")]  # 기준 조건
fig, ax = plt.subplots(figsize=(5.5, 5))
for m in ["Kumo", "TabICLv2", "XGB-default", "XGB-tuned", "CatBoost-tuned", "LogReg"]:
    d = base[base["model"] == m]
    if d.empty: continue
    bins = pd.cut(d["p"], np.linspace(0, 1, 11), include_lowest=True)
    g = d.groupby(bins, observed=True).agg(p=("p", "mean"), y=("y", "mean"))
    ax.plot(g["p"], g["y"], "o-", label=m, color=PALETTE[m])
ax.plot([0, 1], [0, 1], "k--", lw=0.8); ax.set_xlabel("predicted probability"); ax.set_ylabel("observed positive rate")
ax.set_title("Reliability diagram (Part B base condition)"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT_DIR / "figures" / f"calibration_{MODE}.png", dpi=160); plt.show()

### 6-4. 비용 — 준비·튜닝·학습·추론 합계
FM은 튜닝이 없는 대신 추론에 GPU 연산이 들어가고, 튜닝 부스팅은 탐색 비용이 대부분입니다. 같은 축에 놓고 성능과 함께 봅니다.

In [ ]:
cost = R.groupby("model")[["prep_s", "tune_s", "fit_s", "predict_s", "total_s"]].median().loc[ORDER]
display(cost.round(3))
print("FM 1회성 로딩 시간(s):", {k: round(v, 1) for k, v in FM_LOAD_S.items()})
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
cost[["prep_s", "tune_s", "fit_s", "predict_s"]].plot.barh(stacked=True, ax=axes[0], logx=False)
axes[0].set_xlabel("median seconds per task"); axes[0].set_title("Cost breakdown per task")
pareto = R.groupby("model").agg(t=("total_s", "median"), ll=("logloss", "mean")).loc[ORDER]
for m, r in pareto.iterrows():
    axes[1].scatter(r["t"], r["ll"], s=80, color=PALETTE[m]); axes[1].annotate(m, (r["t"], r["ll"]), fontsize=8, xytext=(4, 4), textcoords="offset points")
axes[1].set_xscale("log"); axes[1].set_xlabel("median total seconds (log)"); axes[1].set_ylabel("mean log-loss (↓)")
axes[1].set_title("Accuracy–cost frontier")
plt.tight_layout(); plt.savefig(OUT_DIR / "figures" / f"cost_{MODE}.png", dpi=160); plt.show()

학습 행 수가 늘면 FM은 문맥 전체를 attention으로 처리하므로 비용이 빠르게 커집니다. 부스팅은 튜닝 비용이 대부분입니다. **두 곡선이 교차하는 행 수**가 비용 관점의 전환점입니다.

In [ ]:
sz = R[(R["part"] == "B") & (R["axis"] == "size")].copy()
sz["n"] = sz["level"].astype(int)
fig, ax = plt.subplots(figsize=(7, 4.5))
for m in ["Kumo", "TabICLv2", "XGB-tuned", "LGBM-tuned", "CatBoost-tuned", "CatBoost-default"]:
    g = sz[sz["model"] == m].groupby("n")["total_s"].median()
    ax.plot(g.index, g.values, "o-", label=m, color=PALETTE[m])
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("train rows"); ax.set_ylabel("median total seconds")
ax.set_title(f"Cost vs train size ({DEVICE.type})"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT_DIR / "figures" / f"cost_by_size_{MODE}.png", dpi=160); plt.show()

## 7. 결정 가이드
조건마다 **Kumo vs 부스팅 6종 최고값**의 승률과 평균 차이를 정리합니다.
- **FM 권장**: log-loss 승률 ≥ 60%이고 평균 Δ < 0
- **부스팅 권장**: 승률 ≤ 40%이고 평균 Δ > 0
- **비슷함**: 그 외. 이때는 비용, 운영(GPU 필요 여부), 해석 가능성으로 결정합니다.
- 한 셀의 표본(데이터셋 × seed)이 `MIN_N`개 미만이면 **(잠정)**을 붙입니다. quick 모드의 판단은 모두 잠정입니다.

In [ ]:
MIN_N = 6
def verdict(win, d, n):
    v = "FM 권장" if (win >= .6 and d < 0) else ("부스팅 권장" if (win <= .4 and d > 0) else "비슷함 → 비용·운영으로 결정")
    return v if n >= MIN_N else f"{v} (잠정, n={n})"
k = BD[BD["model"] == "Kumo"]
guide = (k.groupby(["axis", "level"])
         .agg(n=("d_logloss", "size"), win_logloss=("d_logloss", lambda s: (s < 0).mean()),
              d_logloss=("d_logloss", "mean"), win_auc=("d_auc", lambda s: (s > 0).mean()), d_auc=("d_auc", "mean"))
         .reset_index())
guide["axis_order"] = guide["axis"].map({a: i for i, a in enumerate(B_AXES)})
guide["lv_order"] = guide.apply(lambda r: level_order(r["axis"]).index(r["level"]), axis=1)
guide = guide.sort_values(["axis_order", "lv_order"]).drop(columns=["axis_order", "lv_order"])
guide["판단"] = [verdict(w, d, n) for w, d, n in zip(guide["win_logloss"], guide["d_logloss"], guide["n"])]
guide.to_csv(OUT_DIR / f"decision_guide_{MODE}.csv", index=False)
guide.round(3)

In [ ]:
# Kumo가 진 과제: 부스팅 6종 중 하나라도 log-loss가 더 낮은 경우 → 반례가 어디서 나오는지 확인
win_gb = B[B["model"].isin(GBDT_ALL)].loc[lambda d: d.groupby(TASK_COLS)["logloss"].idxmin()][TASK_COLS + ["model"]]
lost = (BD[(BD["model"] == "Kumo") & (BD["d_logloss"] > 0)]
        .merge(win_gb.rename(columns={"model": "winner"}), on=TASK_COLS)
        [TASK_COLS + ["logloss", "best_logloss", "winner", "auc", "best_auc"]])
print(f"Kumo가 진 과제: {len(lost)} / {BD[BD['model'] == 'Kumo'].shape[0]}")
lost.round(4)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 2.6), gridspec_kw=dict(width_ratios=[len(v) for v in B_AXES.values()]))
for ax, axis in zip(axes, B_AXES):
    g = guide[guide["axis"] == axis].set_index("level").reindex(level_order(axis))
    sns.heatmap(g[["d_logloss"]].T, ax=ax, cmap="RdBu", center=0, annot=True, fmt=".3f", cbar=False,
                vmin=-0.05, vmax=0.05)
    ax.set_title(AXIS_LABEL[axis]); ax.set_yticklabels(["Δ log-loss"], rotation=0); ax.set_xlabel("")
plt.suptitle("Kumo − best of 6 GBDT variants, Δ log-loss (blue = Kumo better)", y=1.08)
plt.savefig(OUT_DIR / "figures" / f"decision_map_{MODE}.png", dpi=160, bbox_inches="tight"); plt.show()

## 8. 해석할 때 주의할 점
- **"추가 학습이 없다" ≠ "레이블이 필요 없다"**: FM도 레이블된 학습 행을 문맥으로 받습니다. 다른 점은 과제별 가중치를 업데이트하지 않는다는 것입니다.
- **사전학습 범위**: Kumo는 최대 6만 행·100열의 합성 표에서 학습되었습니다. 이 범위를 크게 벗어나거나 테스트 분포가 문맥과 다르면 성능이 떨어질 수 있다고 공식 자료에 적혀 있습니다.
- **튜닝 예산**: 부스팅의 순위는 `tune_trials`에 따라 달라집니다. 예산을 바꿔 다시 돌려 보면, FM과 동등해지는 데 필요한 튜닝 비용을 추정할 수 있습니다.
- **기준선의 편향**: "부스팅 6종 최고값"은 테스트 셋 기준으로 사후에 고른 값이라 부스팅에 유리합니다. 이 기준을 이기면 FM 우위의 근거가 강해지고, 지면 결론을 해석할 때 이 점을 감안해야 합니다.
- **`cat_frac` 축**: 구간화로 정보가 손실되므로, 순수한 "범주형 비율" 효과보다 과장되거나 축소될 수 있습니다. Part A의 자연 상태 범주형 비율과 함께 보세요.
- **통계적 신뢰도**: quick 모드는 seed 1개, 데이터셋 2개입니다. 결론은 full 모드(seed 3개)의 승률과 SE로 내리세요.